You can use the Anderson–Darling test to examine whether the weights of an ML model are consistent with a normal distribution.

One important point first:

Do not combine all model weights from all layers into one giant array and test it blindly. Different layers can have different means, variances, and distributions. A better approach is to test each layer separately.

Below is a complete example using a Scikit-learn MLP.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier
from scipy.stats import anderson


# ============================================================
# 1. Create example ML dataset
# ============================================================

X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=42
)


# ============================================================
# 2. Train an ML model
# ============================================================

model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

model.fit(X, y)


# ============================================================
# 3. Anderson-Darling test function
# ============================================================

def test_weights_normality(weights, layer_name, alpha=0.05):

    # Flatten the weight matrix
    weights = np.asarray(weights).flatten()

    # Remove NaN / infinity if present
    weights = weights[np.isfinite(weights)]

    # Anderson-Darling normality test
    result = anderson(
        weights,
        dist="norm"
    )

    print("\n" + "=" * 60)
    print(f"Layer: {layer_name}")
    print("=" * 60)

    print(f"Number of weights : {len(weights)}")
    print(f"Mean              : {np.mean(weights):.6f}")
    print(f"Std                : {np.std(weights):.6f}")
    print(f"AD statistic       : {result.statistic:.6f}")

    print("\nCritical values:")

    # scipy provides significance levels such as
    # 15%, 10%, 5%, 2.5%, 1%
    for significance, critical_value in zip(
        result.significance_level,
        result.critical_values
    ):

        print(
            f"{significance:5.1f}% significance "
            f"-> critical value = {critical_value:.6f}"
        )

    # Find the critical value corresponding to alpha
    significance_levels = np.asarray(result.significance_level)

    target_percent = alpha * 100

    # Find closest available significance level
    index = np.argmin(
        np.abs(significance_levels - target_percent)
    )

    critical_value = result.critical_values[index]

    print(f"\nChosen alpha      : {alpha}")
    print(f"Critical value    : {critical_value:.6f}")

    # ========================================================
    # Decision
    # ========================================================

    if result.statistic > critical_value:

        print("\nDecision: REJECT H0")

        print(
            "There is statistically significant evidence "
            "that this layer's weights do not follow the "
            "specified normal distribution."
        )

        return {
            "layer": layer_name,
            "ad_statistic": result.statistic,
            "critical_value": critical_value,
            "decision": "Reject H0"
        }

    else:

        print("\nDecision: FAIL TO REJECT H0")

        print(
            "There is insufficient statistical evidence "
            "to conclude that this layer's weights differ "
            "from a normal distribution."
        )

        return {
            "layer": layer_name,
            "ad_statistic": result.statistic,
            "critical_value": critical_value,
            "decision": "Fail to reject H0"
        }


# ============================================================
# 4. Test every neural-network layer
# ============================================================

results = []

for i, weights in enumerate(model.coefs_):

    result = test_weights_normality(
        weights,
        layer_name=f"Layer {i + 1}",
        alpha=0.05
    )

    results.append(result)


# ============================================================
# 5. Summary
# ============================================================

print("\n\n")
print("=" * 70)
print("MODEL WEIGHT NORMALITY SUMMARY")
print("=" * 70)

for result in results:

    print(
        f"{result['layer']:10s} | "
        f"AD = {result['ad_statistic']:.4f} | "
        f"Critical = {result['critical_value']:.4f} | "
        f"{result['decision']}"
    )


Layer: Layer 1
Number of weights : 640
Mean              : -0.006179
Std                : 0.273349
AD statistic       : 0.654994

Critical values:
 15.0% significance -> critical value = 0.572000
 10.0% significance -> critical value = 0.652000
  5.0% significance -> critical value = 0.782000
  2.5% significance -> critical value = 0.912000
  1.0% significance -> critical value = 1.085000

Chosen alpha      : 0.05
Critical value    : 0.782000

Decision: FAIL TO REJECT H0
There is insufficient statistical evidence to conclude that this layer's weights differ from a normal distribution.

Layer: Layer 2
Number of weights : 512
Mean              : 0.023951
Std                : 0.303003
AD statistic       : 1.433617

Critical values:
 15.0% significance -> critical value = 0.572000
 10.0% significance -> critical value = 0.651000
  5.0% significance -> critical value = 0.781000
  2.5% significance -> critical value = 0.911000
  1.0% significance -> critical value = 1.084000

Chosen alpha  

Anderson–Darling Two-Sample Test

The Anderson–Darling two-sample test (AD 2-sample test) is a non-parametric hypothesis test used to determine whether two independent samples come from the same probability distribution.

For your portfolio example, it can answer:

"Do Strategy A and Strategy B have statistically different return distributions?"

The major advantage over the KS test is that the Anderson–Darling test gives more weight to differences in the tails.

In [2]:
import numpy as np
from scipy.stats import anderson_ksamp


# ============================================================
# Strategy returns
# ============================================================

strategy_a = np.array([
    0.012, 0.008, 0.015, -0.003, 0.010,
    0.007, -0.002, 0.013, 0.009, 0.011,
    0.006, 0.010, 0.004, 0.012, 0.008,
    0.014, 0.006, -0.004, 0.011, 0.009
])

strategy_b = np.array([
    0.005, 0.010, 0.004, 0.002, 0.006,
    0.003, 0.001, 0.005, 0.007, 0.004,
    0.003, 0.006, 0.005, 0.004, 0.006,
    0.002, 0.004, 0.003, 0.005, 0.004
])


# ============================================================
# Anderson-Darling TWO-SAMPLE test
# ============================================================

result = anderson_ksamp([
    strategy_a,
    strategy_b
])


# ============================================================
# Results
# ============================================================

print("Anderson-Darling Two-Sample Test")
print("---------------------------------")

print("AD statistic :", result.statistic)
print("p-value      :", result.pvalue)

alpha = 0.05


# ============================================================
# Decision
# ============================================================

if result.pvalue <= alpha:

    print("\nDecision: Reject H0")

    print(
        "There is statistically significant evidence "
        "that Strategy A and Strategy B have different "
        "return distributions."
    )

else:

    print("\nDecision: Fail to reject H0")

    print(
        "There is insufficient statistical evidence "
        "to conclude that the return distributions "
        "of Strategy A and Strategy B differ."
    )

Anderson-Darling Two-Sample Test
---------------------------------
AD statistic : 8.281448478423616
p-value      : 0.001

Decision: Reject H0
There is statistically significant evidence that Strategy A and Strategy B have different return distributions.


/tmp/ipykernel_1010/2407456361.py:28: UserWarning: p-value floored: true value smaller than 0.001. Consider specifying `method` (e.g. `method=stats.PermutationMethod()`.)
  result = anderson_ksamp([


Example decision

Suppose you get:
AD statistic = 3.21
p-value      = 0.012
Then:
0.012<0.05

Therefore:

Reject H0
Your correct conclusion is:

There is statistically significant evidence that Strategy A and Strategy B have different return distributions at the 5% significance level.
Don't say:

❌ Strategy A is better than Strategy B.

The AD test only tells you that the distributions differ.